# 🧪 Pruebas Unitarias y CI/CD con GitHub Actions
## Algoritmos y Estructuras de Datos I — UADE

---

Un programa que funciona hoy puede romperse mañana cuando alguien modifica una función.  
Las **pruebas unitarias** son la red de seguridad que detecta esas roturas antes de que lleguen a producción.

### Hoja de ruta

| Bloque | Tema |
|---|---|
| 1 | ¿Qué son las pruebas unitarias y por qué importan? |
| 2 | `pytest` — instalación y estructura básica |
| 3 | Tests sobre `utilities.py` — validaciones con regex |
| 4 | Tests sobre `services.py` — lógica de negocio con mocking |
| 5 | Pruebas parametrizadas y fixtures |
| 6 | CI/CD con GitHub Actions |
| 7 | Buenas prácticas |

> 💡 A lo largo del notebook vamos a testear el código real del Demo TPO:  
> `utilities.py`, `services.py` y `storage.py`.


---
## Bloque 1 — ¿Qué son las pruebas unitarias y por qué importan?

Una **prueba unitaria** verifica que una función hace exactamente lo que se espera, de forma aislada y automática.

### El problema que resuelven

Imaginá que `validar_precio_unitario()` funciona perfecto hoy.  
Mañana alguien modifica el patrón regex para aceptar precios negativos.  
Sin tests, ese bug llega a producción. Con tests, se detecta en segundos.

### Las tres preguntas de un test

| Pregunta | Ejemplo |
|---|---|
| ¿Qué función estoy probando? | `validar_precio_unitario()` |
| ¿Qué entrada le doy? | `"99.99"` |
| ¿Qué resultado espero? | `True` |

### Qué testear y qué no

| ✅ Testear | ❌ No testear |
|---|---|
| Funciones de validación (`utilities.py`) | El menú de `main.py` (difícil de automatizar) |
| Lógica de negocio (`services.py`) | El `input()` del usuario |
| Casos borde (string vacío, precio 0) | Librerías externas (tabulate, re) |
| Manejo de errores (`raise`, `ValueError`) | Código trivial (getters simples) |

### El patrón AAA — Arrange, Act, Assert

Todo test bien escrito tiene tres partes:

```python
def test_validar_precio_valido():
    # Arrange — preparar los datos de entrada
    precio = "99.99"

    # Act — ejecutar la función bajo prueba
    resultado = validar_precio_unitario(precio)

    # Assert — verificar el resultado esperado
    assert resultado == True
```


### Evidencia académica: ¿por qué las pruebas son el factor #1?

La importancia de testear no es solo intuición — está respaldada por evidencia empírica.

Shahin, Babar y Zhu (2017) realizaron una revisión sistemática de 69 estudios sobre prácticas de CI/CD publicados entre 2004 y 2016. Su principal hallazgo para el notebook:

> **"Testing (esfuerzo y tiempo)"** es el factor crítico número uno para el éxito de CI/CD, citado en **27 de los 69 estudios revisados (39.1%)**.

El estudio también revela que los problemas de testing más frecuentes en equipos reales son exactamente los que cubrimos en este notebook:
- Tests de larga duración que frenan el pipeline
- Baja cobertura de tests
- Tests inestables (que a veces pasan y a veces fallan sin cambios en el código)
- Falta de automatización

> *Fuente: Shahin, M., Babar, M. A., & Zhu, L. (2017). Continuous integration, delivery and deployment: A systematic review. IEEE Access, 5, 3909–3943. https://doi.org/10.1109/ACCESS.2017.2685629*


---
## Bloque 2 — `pytest`: instalación y estructura

### ¿Por qué pytest y no unittest?

| Característica | `unittest` | `pytest` |
|---|---|---|
| Sintaxis | Clases y métodos | Funciones simples |
| `assert` | Métodos especiales (`assertEqual`) | `assert` nativo de Python |
| Parametrización | Compleja | `@pytest.mark.parametrize` |
| Industria | Legacy | Estándar actual |

### Instalación

```bash
pip install pytest
python -m pytest --version
```

### Estructura del proyecto con tests

```
proyecto/
├── src/
│   ├── core/
│   │   ├── services.py
│   │   └── storage.py
│   ├── interface/
│   │   └── inputHandler.py
│   ├── shared/
│   │   └── utilities.py
│   └── main.py
├── test/
│   ├── __init__.py          ← vacío, hace que test/ sea un paquete
│   ├── test_utilities.py    ← tests de validaciones
│   └── test_services.py     ← tests de lógica de negocio
├── data/
│   └── productos.json
└── pytest.ini               ← configuración de pytest
```

### `pytest.ini` — configuración mínima

```ini
[pytest]
testpaths = test
pythonpath = src
```

- `testpaths`: dónde buscar los tests
- `pythonpath`: agrega `src/` al path para que los imports funcionen

### Convenciones de nomenclatura

| Elemento | Convención | Ejemplo |
|---|---|---|
| Archivo de tests | `test_<módulo>.py` | `test_utilities.py` |
| Función de test | `test_<función>_<caso>` | `test_validar_precio_valido` |
| Un test = | Un solo comportamiento | No mezclar casos en un mismo test |


In [1]:
# Instalamos pytest en el entorno del notebook
import subprocess
result = subprocess.run(["pip", "install", "pytest", "--quiet"], capture_output=True, text=True)
print(result.stdout or "pytest ya instalado.")

import pytest
print(f"pytest versión: {pytest.__version__}")

pytest ya instalado.
pytest versión: 8.4.2


---
## Bloque 3 — Tests sobre `utilities.py`

`utilities.py` contiene las funciones de validación con regex.  
Son las más fáciles de testear: entrada → True o False, sin dependencias externas.

### El módulo bajo prueba


In [3]:
# ── src/shared/utilities.py ──────────────────────────────────────
import re

def validar_precio_unitario(precio):
    patron = r"[0-9]+(\.[0-9]{1,2})?"
    return bool(re.fullmatch(patron, precio.strip()))

print("utilities.py cargado.")

utilities.py cargado.


### Tests de `validar_precio_unitario()`

Cubrimos los casos más importantes:
- **Happy path**: valores válidos que deben retornar `True`
- **Edge cases**: casos borde como `"0"`, precio exacto
- **Sad path**: valores inválidos que deben retornar `False`


In [ ]:
import pytest

# ── test/test_utilities.py — validar_precio_unitario ─────────────

def test_validar_precio_entero_valido():
    # Arrange
    precio = "350"
    # Act
    resultado = validar_precio_unitario(precio)
    # Assert
    assert resultado == True

def test_validar_precio_decimal_valido():
    # Arrange + Act + Assert en una línea cuando es simple
    assert validar_precio_unitario("99.99") == True

def test_validar_precio_vacio_invalido():
    assert validar_precio_unitario("") == False

def test_validar_precio_letras_invalido():
    assert validar_precio_unitario("abc") == False

def test_validar_precio_negativo_invalido():
    assert validar_precio_unitario("-50") == False


# Ejecutamos los tests directamente en el notebook
resultados = []
tests = [
    test_validar_precio_entero_valido,
    test_validar_precio_decimal_valido,
    test_validar_precio_letras_invalido,
    test_validar_precio_negativo_invalido,
    test_validar_precio_vacio_invalido,
]
for t in tests:
    try:
        t()
        print(f"  ✅ {t.__name__}")
    except AssertionError as e:
        print(f"  ❌ {t.__name__} — {e}")

  ✅ test_validar_precio_entero_valido
  ✅ test_validar_precio_decimal_valido
  ✅ test_validar_precio_letras_invalido
  ✅ test_validar_precio_negativo_invalido
  ✅ test_validar_precio_vacio_invalido


### Tests de `validar_nombre()` y `validar_categoria()`


In [ ]:
# ── test/test_utilities.py — validar_nombre ──────────────────────
# Para completar en clase

# ── test/test_utilities.py — validar_categoria ───────────────────

# Para completar en clase

tests = [
    # Completar con los tests que hayas creado
]
for t in tests:
    try:
        t()
        print(f"  ✅ {t.__name__}")
    except AssertionError:
        print(f"  ❌ {t.__name__}")

---
## Bloque 4 — Tests sobre `services.py` con mocking

`services.py` depende de `storage.py` para leer y escribir en disco.  
En un test **no queremos tocar el disco**: sería lento, dejaría archivos basura y haría los tests frágiles.

La solución es el **mocking**: reemplazar `storage.read_data` y `storage.write_data` por versiones falsas que operan en memoria.

### ¿Qué es un mock?

Un mock es un objeto que **simula** el comportamiento de otro sin ejecutarlo realmente.

```
Test                    services.py              storage.py (real)
  │                         │                         │
  │── llama ──────────────► │                         │
  │                         │── normalmente ─────────►│── lee disco
  │                         │                         │
  │  con mock:              │                         │
  │── llama ──────────────► │                         │
  │                         │── mock ────────────────►│ (no existe)
  │                         │◄─ retorna lista fija ───│
```

Usamos `unittest.mock.patch` para reemplazar funciones durante el test.


In [1]:
import json

def read_data(ruta):
    with open(ruta, "r", encoding="utf-8") as file:
        return json.load(file)


def write_data(ruta, datos):
    with open(ruta, "w", encoding="utf-8") as file:
        json.dump(datos, file, indent=4, ensure_ascii=False)
    return True


In [ ]:
import re

# ── Funciones de services.py ──────────────────────────────────────
RUTA = "data/productos.json"

def generar_id_producto(productos):
    return max((p["id_producto"] for p in productos), default=0) + 1

def crear_producto(producto):
    productos = read_data(RUTA)
    producto["id_producto"] = generar_id_producto(productos)
    productos.append(producto)
    write_data(RUTA, productos)
    return True

def get_producto_by_id(id_producto):
    productos = read_data(RUTA)
    productos_found = [producto for producto in productos if re.fullmatch(str(id_producto), str(producto["id_producto"]))]
    return productos_found[0] if productos_found else None

print("services.py cargado.")

services.py cargado.


### Tests de `crear_producto()` con `patch`

`patch("core.storage.read_data")` reemplaza la función real por un mock durante el test.  
`patch("__main__.read_data")` en este notebook usamos __main__ en lugar de storage.  
`mock_read.return_value = [...]` define qué retorna el mock cuando se lo llama.


In [7]:
from unittest.mock import patch

# La lista que el mock va a "devolver" cuando se llame a read_data
productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 350},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul", "precio_unitario": 500}
    ]

# Declaramos la función que testea a crear_producto()
def test_crear_producto():
    # Arrange
    nuevo_producto = {"nombre": "Hojas A4", "categoria": "Papel", "precio_unitario": 500.0}

    # usamos with con patch, de manera que cuando crea_producto invoque a 
    # read_data / write_data use el método dummy en lugar del real
    # y asi no afectar el archivo json en disco.
    with patch("__main__.read_data", return_value=productos.copy()) as mock_read, \
        patch("__main__.write_data", return_value=True) as mock_write:

        # Act
        resultado = crear_producto(nuevo_producto)

        # Assert — ¿retornó True?
        assert resultado == True

        # Assert — ¿write_data fue llamado con la lista correcta?
        productos_actualizados = mock_write.call_args[0][1]  # segundo argumento de write_data
        assert len(productos_actualizados) == 3
        assert productos_actualizados[-1]["id_producto"] == 3  # ID generado automáticamente
        assert productos_actualizados[-1]["nombre"] == "Hojas A4"


try:
    test_crear_producto()
    print(f"  ✅ {test_crear_producto.__name__}")
except Exception as e:
    print(f"  ❌ {test_crear_producto.__name__} — {e}")

  ✅ test_crear_producto


### Tests de `get_producto_by_id()`


In [10]:
from unittest.mock import patch

# La lista que el mock va a "devolver" cuando se llame a read_data
productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 350},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul", "precio_unitario": 500}
    ]

# Declaramos la función que testea a get_producto_by_id() que retorna match
def test_get_producto_by_id_encontrado():
    with patch("__main__.read_data", return_value=productos.copy()) as mock_read:
        # Arrange
        id_producto = 2
        # Act
        resultado = get_producto_by_id(id_producto)
        # Assert — ¿retornó True?
        assert resultado is not None
        assert resultado["nombre"] == "Bolígrafo azul"

# Declaramos la función que testea a get_producto_by_id() que no retorna match
def test_get_producto_by_id_no_encontrado():
    with patch("__main__.read_data", return_value=productos.copy()) as mock_read:
        # Arrange
        id_producto = 99
        # Act
        resultado = get_producto_by_id(id_producto)
        # Assert — ¿retornó True?
        assert resultado is None

tests = [
    test_get_producto_by_id_encontrado,
    test_get_producto_by_id_no_encontrado,
]
for t in tests:
    try:
        t()
        print(f"  ✅ {t.__name__}")
    except (AssertionError, Exception) as e:
        print(f"  ❌ {t.__name__} — {e}")

  ✅ test_get_producto_by_id_encontrado
  ✅ test_get_producto_by_id_no_encontrado


---
## Bloque 5 — Pruebas parametrizadas

### `@pytest.mark.parametrize` — un test, muchos casos

Cuando tenemos varios casos de prueba para la misma función, en lugar de escribir un test por caso usamos `parametrize`:


In [ ]:
import pytest

# ── Sin parametrize: repetitivo ───────────────────────────────────
def test_precio_350():     assert validar_precio_unitario("350")    == True
def test_precio_99_99():   assert validar_precio_unitario("99.99")  == True
def test_precio_abc():     assert validar_precio_unitario("abc")    == False
# ... y así para cada caso

# ── Con parametrize: limpio y escalable ───────────────────────────
@pytest.mark.parametrize("precio, esperado", [
    # Happy path
    ("350",     True),
    ("99.99",   True),
    ("1200.5",  True),
    ("0",       True),
    # Sad path
    ("abc",     False),
    ("-50",     False),
    ("99.999",  False),
    ("",        False),
    (".99",     False),
])
def test_validar_precio_parametrizado(precio, esperado):
    assert validar_precio_unitario(precio) == esperado

# En el notebook lo ejecutamos manualmente
casos = [
    ("350", True), ("99.99", True), ("1200.5", True), ("0", True),
    ("abc", False), ("-50", False), ("99.999", False), ("", False), (".99", False),
]
print("test_validar_precio_parametrizado:")
for precio, esperado in casos:
    resultado = validar_precio_unitario(precio)
    ok = resultado == esperado
    print(f"  {'✅' if ok else '❌'} precio='{precio}' → esperado={esperado}, obtenido={resultado}")

### Cómo ejecutar pytest desde la terminal

```bash
# Desde la raíz del proyecto
python -m pytest                    # ejecuta todos los tests
python -m pytest -v                 # verbose: muestra cada test
python -m pytest test/test_utilities.py   # solo un archivo
python -m pytest -k "precio"        # solo tests cuyo nombre contiene "precio"
python -m pytest --tb=short         # traceback corto en caso de fallo
```

### Interpretando la salida

```
collected 14 items

test/test_utilities.py::test_validar_precio_entero_valido   PASSED  ✅
test/test_utilities.py::test_validar_precio_letras_invalido PASSED  ✅
test/test_services.py::test_crear_producto_exitoso          PASSED  ✅
test/test_services.py::test_crear_producto_nombre_duplicado FAILED  ❌

FAILURES
─────────────────────────────────────────────────────────────
test/test_services.py::test_crear_producto_nombre_duplicado
  AssertionError: Debería haber lanzado ValueError

14 passed, 1 failed in 0.42s
```

Un test en **FAILED** significa que el código no se comporta como esperábamos. Hay que arreglar el código (o el test si estaba mal escrito), nunca ignorar el fallo.


---
## Bloque 6 — CI/CD con GitHub Actions

### ¿Qué es CI/CD?

| Sigla | Nombre | Qué hace |
|---|---|---|
| **CI** | Continuous Integration | Ejecuta los tests automáticamente cada vez que alguien sube código |
| **CD** | Continuous Delivery/Deployment | Despliega el código a producción si los tests pasan |

### Fundamento académico del CI — ¿por qué automatizar?

La revisión sistemática de Shahin et al. (2017) identificó **7 factores críticos** para el éxito de las prácticas continuas. Los dos más relevantes para entender el valor de GitHub Actions son:

| Rank | Factor | Papers que lo citan | Relación con GitHub Actions |
|---|---|---|---|
| #1 | Testing (esfuerzo y tiempo) | 27 (39.1%) | El pipeline ejecuta los tests automáticamente en cada push |
| #2 | Team awareness & transparency | 24 (34.7%) | Los badges y logs hacen visible el estado del proyecto a todo el equipo |

El estudio también destaca que **Git/GitHub** fue uno de los sistemas de control de versiones más utilizados en los pipelines de despliegue relevados en la literatura, junto con Jenkins como servidor de CI.


Para el proyecto usamos **CI**: cada `git push` o Pull Request dispara los tests automáticamente en GitHub.

### El flujo sin CI vs con CI

```
Sin CI:
  git push → nadie sabe si rompió algo → el bug llega a main

Con CI:
  git push → GitHub Actions ejecuta pytest → ✅ merge permitido
                                           → ❌ merge bloqueado hasta arreglar
```

### GitHub Actions

GitHub Actions es el sistema de automatización integrado en GitHub.  
Se configura con archivos `.yml` en la carpeta `.github/workflows/`.

```
proyecto/
├── .github/
│   └── workflows/
│       └── ci.yml      ← el pipeline de CI
├── src/
├── test/
└── ...
```

### El archivo `ci.yml`

```yaml
# .github/workflows/ci.yml

name: CI — Pruebas Unitarias        # nombre del workflow (aparece en GitHub)

on:                                  # cuándo se dispara
  push:                              # en cada push
    branches: [ main, develop ]      # solo en estas ramas
  pull_request:                      # y en cada Pull Request
    branches: [ main ]

jobs:
  test:                              # nombre del job
    runs-on: ubuntu-latest           # sistema operativo del runner

    steps:
      - name: Checkout del código
        uses: actions/checkout@v4    # clona el repositorio

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependencias
        run: |
          pip install pytest
          pip install tabulate

      - name: Ejecutar tests
        run: python -m pytest -v
```

### Cómo interpretarlo paso a paso

| Paso | Qué hace GitHub |
|---|---|
| `checkout` | Clona el código del repo en una máquina virtual |
| `setup-python` | Instala Python 3.11 en esa máquina |
| `pip install` | Instala las dependencias del proyecto |
| `pytest -v` | Ejecuta todos los tests |

Si algún test falla, el job queda en ❌ y GitHub notifica por email.  
Si todos pasan, el job queda en ✅ y el PR puede mergearse.

### Proteger la rama `main`

En GitHub → Settings → Branches → Branch protection rules:

- ✅ **Require status checks to pass before merging**
- ✅ **Require branches to be up to date before merging**

Esto impide que alguien haga merge si los tests fallan, **incluso el dueño del repo**.

### Múltiples versiones de Python (matrix)

Para proyectos que deben funcionar en varias versiones de Python:

```yaml
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]   # ejecuta en las 3 versiones

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install pytest tabulate
      - run: python -m pytest -v
```

Esto genera 3 jobs en paralelo, uno por versión.


### Badges de estado — visibilidad para todo el equipo

Un badge en el `README.md` muestra el estado del pipeline en tiempo real:

```markdown
![CI](https://github.com/usuario/proyecto-artemis/actions/workflows/ci.yml/badge.svg)
```

Esto implementa directamente el **Factor #2** del paper (team awareness): cualquier miembro del equipo ve de un vistazo si el código en `main` está en buen estado.

```
README.md del proyecto:
┌─────────────────────────────────────────────┐
│ 🚀 Sistema CRUD — Demo TPO                  │
│                                             │
│ CI ✅ passing   coverage 87%                │
│                                             │
│ Descripción del proyecto...                 │
└─────────────────────────────────────────────┘
```

### Feedback inmediato — el ciclo virtuoso

```
Developer hace push
        ↓
GitHub Actions ejecuta pytest (~ 30 segundos)
        ↓
✅ Tests pasan  →  merge habilitado  →  confianza alta
❌ Tests fallan →  merge bloqueado   →  bug detectado antes de llegar a main
        ↓
Developer corrige y vuelve a hacer push
```

El paper reporta que los equipos que no tienen feedback rápido de sus tests terminan acumulando deuda técnica y perdiendo confianza en el proceso de release.


In [ ]:
# Generamos el archivo ci.yml directamente desde el notebook
import os

os.makedirs(".github/workflows", exist_ok=True)

ci_yml = """name: CI — Pruebas Unitarias

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout del código
        uses: actions/checkout@v4

      - name: Configurar Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependencias
        run: |
          pip install pytest
          pip install tabulate

      - name: Ejecutar tests con pytest
        run: python -m pytest -v
"""

with open(".github/workflows/ci.yml", "w") as f:
    f.write(ci_yml)

print("✅ Archivo .github/workflows/ci.yml generado.")
print()
print("Próximos pasos:")
print("  1. git add .github/")
print("  2. git commit -m 'ci: agregar workflow de pruebas unitarias'")
print("  3. git push")
print("  4. Ir a GitHub → pestaña Actions para ver el pipeline en ejecución")

---
## Bloque 7 — Buenas prácticas

### Los tres tipos de test que debería tener todo módulo

| Tipo | Qué prueba | Ejemplo |
|---|---|---|
| **Happy path** | El caso exitoso normal | precio válido retorna True |
| **Sad path** | Entradas inválidas | precio con letras retorna False |
| **Edge case** | Casos borde | string vacío, precio 0, lista vacía |

### Errores comunes a evitar

| ❌ Error | ✅ Corrección |
|---|---|
| Un test que prueba muchas cosas | Un test = un solo comportamiento |
| Tests que dependen del orden de ejecución | Cada test es independiente |
| Tests que usan archivos reales en disco | Usar mocking para storage |
| Nombres genéricos: `test_1`, `test_2` | Nombres descriptivos: `test_crear_producto_nombre_duplicado` |
| Tests que nunca fallan (siempre `assert True`) | Verificar que el test falla cuando el código falla |

### El ciclo TDD (Test Driven Development)

TDD es una práctica donde **primero escribís el test, después el código**:

```
1. RED   — escribís el test → falla porque la función no existe aún   🔴
2. GREEN — implementás la función mínima para que pase               🟢
3. REFACTOR — mejorás el código sin romper los tests                 🔵
```

No es obligatorio usarlo siempre, pero es útil para funciones complejas como `crear_producto()`.

### Cobertura de tests (`pytest-cov`)

`pytest-cov` mide qué porcentaje del código está cubierto por tests:

```bash
pip install pytest-cov
python -m pytest --cov=src --cov-report=term-missing
```

Salida:
```
Name                    Stmts   Miss  Cover
-------------------------------------------
src/shared/utilities.py    12      0   100%
src/core/services.py       28      4    86%
src/core/storage.py        18      6    67%
```

> 💡 100% de cobertura no garantiza que el código sea correcto, pero menos del 70% en módulos críticos es una señal de alerta.

Para el CI podemos agregar cobertura mínima:

```yaml
- name: Ejecutar tests con cobertura
  run: python -m pytest --cov=src --cov-fail-under=70
```

Esto hace fallar el pipeline si la cobertura cae por debajo del 70%.


---
## 🏋️ Ejercicios prácticos


### Ejercicio 1 — Tests de `validar_categoria()`

Escribí tests parametrizados para `validar_categoria()` cubriendo al menos:
- 3 casos válidos (incluyendo una con espacio y una con acento)
- 3 casos inválidos (número, símbolo, vacío)


In [ ]:
import pytest

@pytest.mark.parametrize("categoria, esperado", [
    # Completá los casos 👇
    # ("Escritura",     ___),
    # ("___",           ___),
])
def test_validar_categoria_parametrizado(categoria, esperado):
    assert validar_categoria(categoria) == esperado

# Ejecutalo con: python -m pytest test/test_utilities.py -v -k "categoria"

### Ejercicio 2 — Test de `update_producto()` con caso no encontrado

Escribí el test para verificar que `update_producto()` retorna `False` cuando el `id_producto` no existe en la lista.


In [ ]:
def test_update_producto_no_encontrado():
    lista = [
        {"id_producto": 1, "nombre": "Lápiz negro HB", "categoria": "Escritura", "precio_unitario": 350.0}
    ]

    with patch.object(storage, "read_data", return_value=lista):
        # Completá: llamá a update_producto con un id que no existe
        # y verificá que retorna False
        pass  # ← reemplazá esto

try:
    test_update_producto_no_encontrado()
    print("✅ test_update_producto_no_encontrado")
except (AssertionError, Exception) as e:
    print(f"❌ test_update_producto_no_encontrado — {e}")

### 🏆 Ejercicio 3 — Desafío: test de `crear_producto()` verifica el ID auto-generado

Escribí un test que verifique que cuando se crea un producto en una lista que ya tiene productos con IDs 1, 3 y 5 (no consecutivos), el nuevo producto recibe el ID 6 (max + 1).

> 💡 Usá `mock_write.call_args[0][1]` para acceder a la lista que se pasó a `write_data` y verificar el ID asignado.


In [ ]:
def test_crear_producto_id_es_max_mas_uno():
    lista_con_gaps = [
        {"id_producto": 1, "nombre": "Producto A", "categoria": "Cat", "precio_unitario": 100.0},
        {"id_producto": 3, "nombre": "Producto B", "categoria": "Cat", "precio_unitario": 200.0},
        {"id_producto": 5, "nombre": "Producto C", "categoria": "Cat", "precio_unitario": 300.0},
    ]
    nuevo = {"nombre": "Producto D", "categoria": "Cat", "precio_unitario": 400.0}

    # Tu solución acá 👇
    pass

try:
    test_crear_producto_id_es_max_mas_uno()
    print("✅ test_crear_producto_id_es_max_mas_uno")
except (AssertionError, Exception) as e:
    print(f"❌ test_crear_producto_id_es_max_mas_uno — {e}")

---
## 📚 Bibliografía

Las siguientes fuentes sustentan el contenido de este notebook:

**Pruebas unitarias — fundamentos y TDD**

- Beck, K. (2002). *Test Driven Development: By Example*. Addison-Wesley.
- Fowler, M. (2018). *Unit Test*. martinfowler.com. https://martinfowler.com/bliki/UnitTest.html
- Okken, B. (2022). *Python Testing with pytest* (2nd ed.). Pragmatic Bookshelf.

**Mocking**

- Fowler, M. (2007). *Mocks Aren't Stubs*. martinfowler.com. https://martinfowler.com/articles/mocksArentStubs.html

**CI/CD — referencia académica principal** ⭐

- Shahin, M., Babar, M. A., & Zhu, L. (2017). Continuous integration, delivery and deployment: A systematic review on approaches, tools, challenges and practices. *IEEE Access*, *5*, 3909–3943. https://doi.org/10.1109/ACCESS.2017.2685629
  - Revisión sistemática de 69 estudios (2004–2016)
  - Identifica "testing" como factor crítico #1 del CI/CD (39.1% de los estudios)
  - Identifica "team awareness & transparency" como factor crítico #2 (34.7%)
  - Confirma Git/GitHub como uno de los sistemas de control de versiones más utilizados en pipelines de despliegue

**CI/CD — referencias de práctica**

- Fowler, M., & Foemmel, M. (2006). *Continuous Integration*. martinfowler.com. https://martinfowler.com/articles/continuousIntegration.html
- Humble, J., & Farley, D. (2010). *Continuous Delivery*. Addison-Wesley.

**Cobertura de código**

- Zhu, H., Hall, P. A. V., & May, J. H. R. (1997). Software unit test coverage and adequacy. *ACM Computing Surveys*, *29*(4), 366–427. https://doi.org/10.1145/267580.267590


---
## 💬 Reflexión

Respondé en esta celda (doble clic para editar):

**1. ¿Por qué usamos mocking en los tests de `services.py` en lugar de leer y escribir archivos reales?**

> _Tu respuesta acá_

**2. ¿Qué ventaja tiene que el pipeline de CI bloquee el merge si los tests fallan?**

> _Tu respuesta acá_

**3. Si `test_crear_producto_exitoso` falla después de que alguien modifica `generar_id()`, ¿qué información te da ese fallo y qué harías?**

> _Tu respuesta acá_
